*Template notebook generated on 2026-10-29 22:14:28*

In [ ]:
import os, sys; sys.path.insert(0, os.path.join(os.path.dirname(globals().get('__vsc_ipynb_file__', '')) or os.getcwd(), 'Files'))
from pset_check import check_answer

In [ ]:
# Enter the email address associated with your Gradescope account.
my_email = "your_email@mit.edu"

# 20.320 Collaborative Modeling Project

## Antibody-drug conjugate trafficking and pharmacokinetic model

**Milestone 1 due:** November 12, 2026, at 11:59 pm.

**Milestone 2 due:** November 24, 2026, at 11:59 pm.

### Background

Antibody-drug conjugates (ADCs) are a class of biologic drugs that have expanded
effective treatment options for patients with cancer. There are 16 ADCs that are
currently FDA-approved for use in the US to treat multiple types of human cancers,
and many more in preclinical and clinical development.

### Objective

Imagine you are a scientist at a biotech or pharma company that is working to
develop ADCs. Your goal is to develop a quantitative model predicting ADC
trafficking and efficacy and use it to understand what makes a safe and effective ADC.

### Project overview

Working with your teammates, generate a model
of ADC trafficking and efficacy (cancer cell killing) using the blockbuster
anti-HER2 ADC **Enhertu** as the basis for your model. The lectures on receptor
trafficking, cell growth and death, and pharmacokinetics will be especially useful
as you build your model, as will the coding notebooks from PSETs 4, 5, and 6.

The project has three parts:

1. **Milestone 1: Base model and dosing analysis.** Diagram and implement the
   system described below, establish its untreated baseline, use the model to
   choose a dose, and explore how the spacing between doses affects the outcome.
2. **Milestone 2: Model modification.** Change an aspect of the ADC system to
   improve efficacy or safety, and analyze the effect.
3. **Final project report.** Bring the base-model analysis and the modification
   together in a written report.

## The system

Your Milestone 1 model should represent the system described below.

The targeted HER2 receptor is synthesized as a surface monomer and exists as both
a monomer and a dimer on cancer cells. HER2 monomer dimerizes reversibly with
association rate constant $k_{\mathrm{on,HER2}}$ and dissociation rate constant
$k_{\mathrm{off,HER2}}$. The Enhertu ADC binds only surface HER2 monomer, with
apparent association and dissociation rate constants $k_{\mathrm{on,ADC}}$ and
$k_{\mathrm{off,ADC}}$. These apparent rate constants describe the observed binding
kinetics of the bivalent antibody to HER2 receptors on cancer cells, without
modeling its two binding arms separately. Free monomer internalizes with
rate constant $k_{er}$, and dimers internalize with rate constant $k_{ec}$. You may
assume that ADC binding does not change the internalization rate of a monomer.

Once internalized, you may assume that the ADC rapidly unbinds from HER2 in the
low-pH environment of the endosome. Internal HER2 monomers and dimers either
recycle to the surface or degrade, with rate constants $k_{\mathrm{rec}}$ and
$k_{\mathrm{deg,HER2}}$. Dimers recycle intact. The unbound ADC stays inside the cell.

Each Enhertu antibody carries eight molecules of the cytotoxic drug deruxtecan,
the ADC's *payload*. For simplicity, model their release from the internalized
ADC as a single first-order step with rate constant $k_{\mathrm{rel,i}}$.
Released endosomal payload either escapes to the cytoplasm or degrades, with rate
constants $k_{\mathrm{esc}}$ and $k_{\mathrm{deg,P}}$. Payload that escapes
accumulates in the cytoplasm, where it also degrades with rate constant
$k_{\mathrm{deg,P}}$. The payload must be released and
must escape the endosome before it can act. Only cytoplasmic payload kills cancer cells.
The killing rate constant is a Monod function of the payload's **cytoplasmic
concentration**. Its maximum value and the concentration that gives half the
maximum are given below. Cancer cells grow with a 14-day
doubling time.

Payload released in circulation, before the ADC reaches cancer cells, is one
source of the side effects patients experience. Because the payload is more toxic
than standard chemotherapy, even low blood concentrations can cause harm, so you
will track the payload concentration in blood alongside tumor killing. Each IV
dose adds loaded ADC (ADC still carrying its payload) to blood. Loaded ADC in
blood is removed by background intact-ADC clearance, by tumor binding, and by circulating payload release with
rate constant $k_{\mathrm{rel,b}}$. As in the endosome, model this release as a
single first-order step that frees all eight payload molecules. Released blood
payload clears with rate constant $k_{\mathrm{clear}}$.

### Modeling assumptions

* Blood is well mixed, and every living cancer cell has the same per-cell state.
* The model treats the initial tumor as made up entirely of cancer cells, with no
  space between them.
* Cell-associated species are tracked in molecules per living cell, and blood
  species in molar concentration. The total number of living cells determines
  how much ADC the tumor binds and takes up from blood.
* Concentrations inside the cell, including the cytoplasmic payload concentration
  that drives killing, are calculated using the whole-cell volume.
* Cell division leaves the average receptor and drug amounts per living cell
  unchanged, and death removes the contents of dying cells.
* Each IV dose is an instantaneous addition of loaded ADC to blood.
* The model ignores the bare antibody left after an ADC releases its payload, even
  though in reality it could still bind HER2 and compete with loaded ADC.
* Released blood payload does not enter tumor cells.

### Model parameters

#### Patient, tumor, and drug properties

| Quantity | Value |
| --- | --- |
| Blood volume | 5 L |
| Patient mass | 70 kg |
| Initial tumor | Sphere, 1 cm diameter |
| Cancer cell | Sphere, 10 μm diameter |
| ADC molecular weight | 156,000 g/mol |
| Payload molecular weight used for reporting | 1000 g/mol, a model assumption |
| Drug-to-antibody ratio | 8 payload molecules per loaded ADC |

#### Kinetic parameters

Use **minutes** for simulation time. Convert reported seconds and days consistently.
The molar HER2 association rate constant must also be converted for per-cell species.

| Parameter | Python name | Given value |
| --- | --- | --- |
| HER2 synthesis | <tt>q_HER2</tt> | 1000 molecules/cell/min |
| HER2 dimer association | <tt>k_on_HER2</tt> | $10^5$ M$^{-1}$ s$^{-1}$ |
| HER2 dimer dissociation | <tt>k_off_HER2</tt> | $3\times10^{-4}$ s$^{-1}$ |
| ADC association to surface monomer | <tt>k_on_ADC</tt> | $6\times10^5$ M$^{-1}$ s$^{-1}$ |
| ADC dissociation | <tt>k_off_ADC</tt> | $3\times10^{-4}$ s$^{-1}$ |
| Monomer and ADC-monomer internalization | <tt>k_er</tt> | 0.003 min$^{-1}$ |
| Dimer internalization | <tt>k_ec</tt> | 0.01 min$^{-1}$ |
| HER2 monomer and intact-dimer recycling | <tt>k_rec</tt> | 0.03 min$^{-1}$ |
| Endosomal HER2 degradation | <tt>k_deg_HER2</tt> | 0.003 min$^{-1}$ |
| Endosomal payload release | <tt>k_rel_i</tt> | 0.06 min$^{-1}$ |
| Circulating payload release | <tt>k_rel_b</tt> | $6.68\times10^{-7}$ min$^{-1}$ (from plasma stability: 2% released in 21 days) |
| Background loaded-ADC clearance | <tt>k_clear_ADC</tt> | 5.7-day half-life |
| Endosomal payload escape | <tt>k_esc</tt> | 0.03 min$^{-1}$ |
| Payload degradation in endosome and cytoplasm | <tt>k_deg_P</tt> | 0.003 min$^{-1}$ |
| Blood payload clearance | <tt>k_clear</tt> | 1 day$^{-1}$ |
| Cancer-cell growth | <tt>mu</tt> | 14-day doubling time |
| Maximum killing rate constant | <tt>delta_max</tt> | 0.15 day$^{-1}$ |
| Cytoplasmic concentration for half-maximal killing | <tt>K_P</tt> | 10 μM |

The maximum killing rate constant is a value chosen for this model, not one fitted
to clinical data, so the selected dose should not be compared directly with
clinical dosing.

### Measuring blood payload exposure

To compare how much free payload the rest of the body is exposed to, use two
measures of the blood payload concentration $[P_b](t)$:

* **Peak concentration:** the highest value of $[P_b]$ during treatment. It
  measures the most intense exposure at any one time.
* **Area under the curve (AUC):** the integral
  $\int_0^{63\,\mathrm{days}} [P_b]\,dt$, in units of concentration × time. It
  measures cumulative exposure. (So, for example, a persistent low concentration can give the
  same AUC as a high concentration that clears quickly.)

The AUC will be calculated by an *accumulator*. This is an extra variable that simply integrates
$[P_b]$ as the simulation runs without affecting the model. The accumulator's final value is the
AUC. Instructions for adding an accumulator appear in a question below.

These measures let you compare regimens and ADC designs with one another. Payload
released in circulation is only one contributor to toxicity, so these measures are imperfect
predictors of patient safety.

## Milestone 1: Base model and dosing analysis

For Milestone 1, you will implement the base model, establish its untreated baseline,
and use the model to find the smallest dose that leaves no more than 5% of the
initial cancer cells after 63 days, while tracking the payload exposure in blood.
You will then use the model explorer to see how the spacing between doses affects
the outcome.

In [ ]:
import os, sys; sys.path.insert(0, os.path.join(os.path.dirname(globals().get('__vsc_ipynb_file__', '')) or os.getcwd(), 'Files'))
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

sys.path.insert(0, os.path.dirname(globals().get('__vsc_ipynb_file__', '')) or os.getcwd())
from modelingTools import (
    AVOGADRO, KineticModel, Compartment, Parameters, Reaction, Impulse, Unit,
    Simulation, plot_simulations,
)
from design_project_helpers import build_model

minutes_per_day = 1440
dose_days = [0.0, 21.0, 42.0]
assessment_day = 63.0
maximum_dose_mg_per_kg = 4.5
target_fraction_remaining = 0.05

_instructor_source = False

from answer_helpers import add_answer
from design_project_helpers import show_dosing_demo

<style class="question-heading-css">
h2.question-heading, h3.question-heading,
h2:has(> .question-heading-marker), h3:has(> .question-heading-marker) {
  color: #7030A0;
  font-weight: bold;
  margin: 0;
  page-break-after: avoid;
  break-after: avoid;
}
h2.question-heading, h2:has(> .question-heading-marker) {
  font-size: 1.5em;
  text-align: center;
}
h3.question-heading, h3:has(> .question-heading-marker) {
  font-size: 1.1em;
}
.question-heading .question-heading-points,
h2:has(> .question-heading-marker) .question-heading-points,
h3:has(> .question-heading-marker) .question-heading-points {
  font-weight: normal;
}
</style>

## <span class="question-heading-marker" data-kind="standalone" data-number="1"></span>Problem 1. Draw the system before coding <span class="question-heading-points">(10 pts — manual)</span>

Draw a diagram of the complete base model. Distinguish blood, the cell surface,
endosomes, and cytoplasm. Label species, transfers, sources, and losses with
consistent symbols and rate constants. Show how cytoplasmic payload influences
living cancer-cell number and where doses enter.

Use this diagram to organize your implementation. A hand-drawn diagram is fine.

In [ ]:
# Run this cell to add your answer (ordered text and image pages, with math/equations).
add_answer("milestone1_system_diagram")

<style class="question-heading-css">
h2.question-heading, h3.question-heading,
h2:has(> .question-heading-marker), h3:has(> .question-heading-marker) {
  color: #7030A0;
  font-weight: bold;
  margin: 0;
  page-break-after: avoid;
  break-after: avoid;
}
h2.question-heading, h2:has(> .question-heading-marker) {
  font-size: 1.5em;
  text-align: center;
}
h3.question-heading, h3:has(> .question-heading-marker) {
  font-size: 1.1em;
}
.question-heading .question-heading-points,
h2:has(> .question-heading-marker) .question-heading-points,
h3:has(> .question-heading-marker) .question-heading-points {
  font-weight: normal;
}
</style>

## <span class="question-heading-marker" data-kind="problem" data-number="2"></span>Problem 2. Build the system model

<style class="question-heading-css">
h2.question-heading, h3.question-heading,
h2:has(> .question-heading-marker), h3:has(> .question-heading-marker) {
  color: #7030A0;
  font-weight: bold;
  margin: 0;
  page-break-after: avoid;
  break-after: avoid;
}
h2.question-heading, h2:has(> .question-heading-marker) {
  font-size: 1.5em;
  text-align: center;
}
h3.question-heading, h3:has(> .question-heading-marker) {
  font-size: 1.1em;
}
.question-heading .question-heading-points,
h2:has(> .question-heading-marker) .question-heading-points,
h3:has(> .question-heading-marker) .question-heading-points {
  font-weight: normal;
}
</style>

### <span class="question-heading-marker" data-kind="subpart" data-number="2.1"></span>2.1 Implement the base model <span class="question-heading-points">(25 pts — autograded)</span>

Implement the system model using the provided library functions. Include every
process in the system, including cell growth, but leave out cell killing, which you
will add in the next part.


The [modeling library guide](Files/modelingTools/documentation.html) and its
[PDF](Files/modelingTools/documentation.pdf) describe the functions used here. <tt>help(Reaction)</tt>
and <tt>help(Simulation)</tt> also work inside the notebook.

Create the <tt>blood</tt> compartment with its volume, for example
<tt>Compartment('blood', volume=blood_volume_L)</tt>. Create the <tt>cell</tt> compartment with
its count variable set to <tt>N</tt> and its volume set to one cell's volume in liters:

```python
network.add_compartment(Compartment(
    'cell', count_variable='N', count_units='cells', volume_per_unit=cell_volume_L
))
```

Define <tt>blood_volume_L</tt> and <tt>cell_volume_L</tt> from the blood volume and cancer
cell diameter in the parameter tables. The cell volume lets <tt>s.molar(...)</tt> convert molecules per cell to M, which you will
need for cell killing.

Adding this compartment creates the count state <tt>N</tt>; do not add it again
with <tt>add_species</tt>. Declare the molecular species using the names below:

| State variable | Meaning | Compartment | Declared units |
| --- | --- | --- | --- |
| <tt>A_b</tt>, <tt>P_b</tt> | Loaded ADC and free payload in blood | <tt>blood</tt> | M |
| <tt>M_s</tt>, <tt>D_s</tt>, <tt>C_s</tt> | Surface monomer, dimer, and ADC-monomer complex | <tt>cell</tt> | #/cell |
| <tt>M_i</tt>, <tt>D_i</tt>, <tt>A_i</tt> | Internal monomer, dimer, and loaded ADC | <tt>cell</tt> | #/cell |
| <tt>P_i</tt>, <tt>P_c</tt> | Endosomal and cytoplasmic free payload | <tt>cell</tt> | #/cell |
| <tt>N</tt> | Living cancer cells | count of <tt>cell</tt> | cells |

Surface, endosomal, and cytoplasmic species all belong to the <tt>cell</tt> compartment.
To track where a molecule is within the cell, the model uses a separate species for each
location, such as <tt>M_s</tt> for surface monomer and <tt>M_i</tt> for internal monomer.

Represent exponential growth of <tt>N</tt> with this syntax:

```python
network.add_reaction(Reaction(
    'Cell growth', 'N', '2 N', forward_rate_constant='mu'
))
```

For a reaction between two per-cell species, such as HER2 dimerization, convert
the molar association rate constant to per-cell units. For binding between a blood
species in M and a receptor in #/cell, keep the association rate constant in
$\mathrm{M}^{-1}\,\mathrm{min}^{-1}$. The binding rate
$k_{\mathrm{on,ADC}}[A_b][M_s]$ is then in molecules per cell per minute.

Call the supplied <tt>build_model</tt> function to turn your network and parameters into
a runnable model, then inspect its equations with <tt>model.print_rate_equations()</tt>.

In [ ]:
# Keep these three names. The automatic answer checks and later parts use them.
network = KineticModel()
parameters = Parameters()

# TODO: add the blood and cell compartments.
# network.add_compartment(Compartment(...))
# network.add_compartment(Compartment(...))

# TODO: declare the molecular species in the table, one call per species.
# network.add_species(...)

# TODO: add reactions to network, and your parameters to parameters.
# You may add cells if that makes your work easier to organize.

# TODO: when network and parameters are complete, replace None with
#       build_model(network, parameters, time_units='min')
model = None

In [ ]:
check_answer("2.1");

<style class="question-heading-css">
h2.question-heading, h3.question-heading,
h2:has(> .question-heading-marker), h3:has(> .question-heading-marker) {
  color: #7030A0;
  font-weight: bold;
  margin: 0;
  page-break-after: avoid;
  break-after: avoid;
}
h2.question-heading, h2:has(> .question-heading-marker) {
  font-size: 1.5em;
  text-align: center;
}
h3.question-heading, h3:has(> .question-heading-marker) {
  font-size: 1.1em;
}
.question-heading .question-heading-points,
h2:has(> .question-heading-marker) .question-heading-points,
h3:has(> .question-heading-marker) .question-heading-points {
  font-weight: normal;
}
</style>

### <span class="question-heading-marker" data-kind="subpart" data-number="2.2"></span>2.2 Add cell killing <span class="question-heading-points">(5 pts — autograded)</span>

Add killing of cancer cells by cytoplasmic payload to your network, then call
<tt>build_model</tt> again so <tt>model</tt> includes it. The cell below supplies the
death reaction. Complete <tt>killing_rate_constant</tt>, then uncomment the reaction
and the call to <tt>build_model</tt>. Make sure your parameter table includes the
killing parameters.

The death reaction uses a custom <tt>rate_function(s, k)</tt> in place of a rate
constant. <tt>s</tt> supplies the current species values, and <tt>k</tt> supplies the
parameters. Dead cells take their per-cell
contents with them, so do not add a second per-cell death loss.

If you rerun this cell, the library replaces your earlier killing reaction and
prints a warning. That is expected.

In [ ]:
def killing_rate_constant(s, k):
    # TODO: return the Monod killing rate constant (delta in the printed equations),
    #       in min^-1, from the cytoplasmic payload concentration.
    return 0.0

# TODO: uncomment once killing_rate_constant is complete.
# network.add_reaction(Reaction(
#     'Cell killing', 'N', {},
#     rate_function=lambda s, k: killing_rate_constant(s, k) * s.N,
#     rate_expression='delta × [N]', rate_expression_latex=r'\delta \, [N]',
#     rate_units='cells/min',
# ))
# model = build_model(network, parameters, time_units='min')
# model.print_rate_equations()

In [ ]:
check_answer("2.2");

<style class="question-heading-css">
h2.question-heading, h3.question-heading,
h2:has(> .question-heading-marker), h3:has(> .question-heading-marker) {
  color: #7030A0;
  font-weight: bold;
  margin: 0;
  page-break-after: avoid;
  break-after: avoid;
}
h2.question-heading, h2:has(> .question-heading-marker) {
  font-size: 1.5em;
  text-align: center;
}
h3.question-heading, h3:has(> .question-heading-marker) {
  font-size: 1.1em;
}
.question-heading .question-heading-points,
h2:has(> .question-heading-marker) .question-heading-points,
h3:has(> .question-heading-marker) .question-heading-points {
  font-weight: normal;
}
</style>

### <span class="question-heading-marker" data-kind="subpart" data-number="2.3"></span>2.3 Add exposure outputs <span class="question-heading-points">(5 pts — autograded)</span>

To compare regimens, you will report the blood payload AUC defined earlier in this
notebook. Add an accumulator named <tt>AUC_P_b</tt> to your
network that integrates $[P_b]$ in M·min, then call <tt>build_model</tt> again so <tt>model</tt>
includes it. An accumulator integrates a quantity as the simulation runs without
feeding back into the model. For a toy molar blood species <tt>B</tt>:

```python
toy.add_accumulator(
    'AUC_B', lambda s, k: s.B, 'M min', 'Integral of blood B'
)
# After simulation, result['AUC_B'][-1] is its AUC in M min.
```

In [ ]:
# TODO: add an accumulator named AUC_P_b that integrates [P_b], then call build_model again.

In [ ]:
check_answer("2.3");

<style class="question-heading-css">
h2.question-heading, h3.question-heading,
h2:has(> .question-heading-marker), h3:has(> .question-heading-marker) {
  color: #7030A0;
  font-weight: bold;
  margin: 0;
  page-break-after: avoid;
  break-after: avoid;
}
h2.question-heading, h2:has(> .question-heading-marker) {
  font-size: 1.5em;
  text-align: center;
}
h3.question-heading, h3:has(> .question-heading-marker) {
  font-size: 1.1em;
}
.question-heading .question-heading-points,
h2:has(> .question-heading-marker) .question-heading-points,
h3:has(> .question-heading-marker) .question-heading-points {
  font-weight: normal;
}
</style>

## <span class="question-heading-marker" data-kind="standalone" data-number="3"></span>Problem 3. Establish the untreated baseline <span class="question-heading-points">(15 pts — manual)</span>

Determine the untreated steady-state surface and endosomal HER2 monomer and dimer
amounts per cell. Use that receptor state as the initial condition for your
simulations, with no ADC or payload initially present.

Simulate the untreated system for 63 days. Show that the receptor amounts per cell
remain steady while the living-cell count follows the given growth law. Report
your receptor values with units, show the relevant time courses, and explain the
checks you used. Choose your own method for finding the steady state.

In [ ]:
# TODO: determine the untreated receptor state and define your initial conditions.
steady_state = {}
initial_values = {}

In [ ]:
# TODO: simulate the untreated system and plot the receptor and cell-count time courses.
untreated = None

In [ ]:
# Run this cell to add your answer (ordered text and image pages, with math/equations).
add_answer("milestone1_untreated")

### Model explorer

The model explorer shows how dose, dosing interval, and each kinetic parameter
affect your model. Run the next cell and click **Launch model explorer**. The
explorer opens in its own window, so keep the notebook kernel running. For each
simulation, it recalculates the untreated receptor steady state for the selected
parameter values, using your receptor initial conditions as a starting guess.
Your initial cell count is retained. Each parameter comparison therefore starts
at its own untreated receptor baseline. If you change your model, close the window
and launch it again.

Set the dose, the interval $\tau$ between doses (doses fall on days $0$, $\tau$, and
$2\tau$), and any parameter values, then click **Update Plots**. To see how much a
quantity matters, choose it from the **parameter sweep** menu and click
**Sweep Parameter** to overlay curves over a range around its current value.
**Reset Defaults** returns to the values at launch.

Use the explorer to get a feel for the model before you search for the smallest
effective dose. This exploration is ungraded, but a later question asks you to
use the explorer for a graded sweep.

In [ ]:
dosing_demo = show_dosing_demo(
    lambda: (globals().get('model'), globals().get('initial_values'))
)

<style class="question-heading-css">
h2.question-heading, h3.question-heading,
h2:has(> .question-heading-marker), h3:has(> .question-heading-marker) {
  color: #7030A0;
  font-weight: bold;
  margin: 0;
  page-break-after: avoid;
  break-after: avoid;
}
h2.question-heading, h2:has(> .question-heading-marker) {
  font-size: 1.5em;
  text-align: center;
}
h3.question-heading, h3:has(> .question-heading-marker) {
  font-size: 1.1em;
}
.question-heading .question-heading-points,
h2:has(> .question-heading-marker) .question-heading-points,
h3:has(> .question-heading-marker) .question-heading-points {
  font-weight: normal;
}
</style>

## <span class="question-heading-marker" data-kind="standalone" data-number="4"></span>Problem 4. Choose a three-dose regimen <span class="question-heading-points">(30 pts — manual)</span>

Find the smallest dose, in increments of **0.1 mg/kg**, that achieves
$N(63\,\mathrm{days})/N_0\leq0.05$. Use three equal doses on days 0, 21, and 42,
with each dose positive and at most 4.5 mg/kg. Use the given kinetic parameters
and the same untreated receptor state and initial tumor for every regimen.

Compare at least **three different dose sizes**, including your selected dose
and the next lower dose. Show that the selected dose meets the target and the
next lower dose fails. Choose your own search method.

Plot loaded ADC in blood, released payload in blood, cytoplasmic payload, and
living-cell fraction over the 63 days. Report the selected dose per administration,
total dose, day-63 cell fraction, blood payload peak in ng/mL, and blood payload AUC
in μg·h/mL.
Explain the efficacy and exposure comparison.

You used <tt>Impulse</tt> in PSET 4. To give three doses, pass three impulses to a
single simulation. An impulse can give the dose as an amount in <tt>'mol'</tt>, which the
simulation converts to blood concentration using the blood volume. (Note that each
impulse represents an addition of Enhertu, not a reset of the concentration.)

<tt>result['A_b']</tt> gives the sampled concentration, <tt>result.in_days()</tt> gives time
in days, and <tt>result['N'][-1]</tt> gives the final cell count. <tt>result['AUC_P_b'][-1]</tt>
is the AUC from your accumulator, in M·min. Convert it to μg·h/mL using the payload
molecular weight.
For a plot, use Matplotlib or <tt>plot_simulations(result, variables=['A_b','N'], time_units='day')</tt>.
<tt>plot_simulations</tt> can also convert units for display. For example,
<tt>unit_conversions={'A_b': (factor, 'μg/mL')}</tt> multiplies <tt>A_b</tt> by <tt>factor</tt> and labels
its axis μg/mL.

In [ ]:
# TODO: find the smallest passing dose in 0.1 mg/kg increments and compare dose sizes.
dose_sizes_mg_per_kg = []
regimen_results = []
comparison_results = []
chosen_result = None

In [ ]:
# Run this cell to add your answer (ordered text and image pages, with math/equations).
add_answer("milestone1_regimen")

<style class="question-heading-css">
h2.question-heading, h3.question-heading,
h2:has(> .question-heading-marker), h3:has(> .question-heading-marker) {
  color: #7030A0;
  font-weight: bold;
  margin: 0;
  page-break-after: avoid;
  break-after: avoid;
}
h2.question-heading, h2:has(> .question-heading-marker) {
  font-size: 1.5em;
  text-align: center;
}
h3.question-heading, h3:has(> .question-heading-marker) {
  font-size: 1.1em;
}
.question-heading .question-heading-points,
h2:has(> .question-heading-marker) .question-heading-points,
h3:has(> .question-heading-marker) .question-heading-points {
  font-weight: normal;
}
</style>

## <span class="question-heading-marker" data-kind="standalone" data-number="5"></span>Problem 5. Explore the dosing interval <span class="question-heading-points">(10 pts — manual)</span>

Your regimen gives three equal doses 21 days apart. Does the spacing matter? The
model explorer can answer that in one sweep. Set the explorer's dose to your
selected dose, choose **Dose interval (days)** from the parameter sweep menu, and
click **Sweep Parameter**. Doses then fall on days $0$, $\tau$, and $2\tau$ for
intervals from about 2 to 31 days, so every curve delivers the same total dose.

Report the interval that leaves the fewest living cells on day 63, and explain
why much shorter and much longer intervals both do worse. Use the cytoplasmic
payload and living-cell plots in your explanation. Think about what limits the
killing rate when cytoplasmic payload is high, how long a dose keeps killing
going after it is given, and what the tumor does in between. Also note how the
blood payload peak and AUC change with the interval. Paste a screenshot of the
explorer window with your answer.

In [ ]:
# Run this cell to add your answer (ordered text and image pages, with math/equations).
add_answer("milestone1_dose_interval")

## Submit Milestone 1

When you have finished the Milestone 1 questions, run the cell below to generate
your submission PDF, <tt>signed_results.json</tt>, and <tt>code.py</tt>, then upload
them to Gradescope.

In [ ]:
if _instructor_source:
    print('Instructor source: build the assignment to generate the runnable template, solution, and submission files.')
else:
    from generate_pset_submission import generate_pset_submission
    generate_pset_submission()

## Milestone 2: Model modification

### How can you change the ADC to improve its efficacy or safety?

Together with your team members, identify an aspect of the model that can be
altered to improve performance of the ADC. You are looking for modifications
that allow the ADC to eliminate cancer cells at a faster rate or at a lower dose
or that would reduce payload toxicity.

Changing the value of a parameter can meet the requirements, but a creative
modification grounded in the biology of the ADC system will tend to score better.
Aim for creativity in your understanding and analysis of the system. The
[project references](#project-references) can help you brainstorm ideas.

Use your Milestone 1 model as the starting point. Explain why you chose your
modification and whether it has biological relevance. Provide a schematic and
any new rate equations required to describe your modified system, and explicitly
state its assumptions.

Explain the results of your model modification. How does the modification affect
the model? Did you have an a priori prediction as to the impact? Do the results
agree with your prediction? What did you learn about what makes a good ADC from
this model modification?

## Final project report

Your final report should bring together the base-model analysis and the
biological modification:

1. Provide a brief written summary of the system, including a schematic diagram
   with rate constants and the rate equations with units. Highlight the
   biological reasoning behind your model. Diagrams and equations can be included
   in an appendix as needed.
2. Assess the temporal performance of the base model without ADC and for at least
   three dose sizes, using the Milestone 1 regimen and target. Report the selected
   dose per administration, total dose, and final cell fraction, show that the next
   lower dose fails, and support the choice with blood payload peak and AUC.
3. Explain your model modification, its biological relevance, and the schematic
   and any new rate equations needed to describe it.
4. Explain the effects of the modification, compare the results with your a
   priori prediction, and discuss what you learned about what makes a good ADC.

Explicitly state all assumptions used in generating and implementing your model.

**Page limit:** do not exceed **8 pages**, double-spaced with **12-point font**.
Figures do not count toward the 8-page limit. This limit applies to the final
project report, not to the automatically generated Milestone 1 submission PDF.

<a id="project-references"></a>

## Project references

### Example ADC trafficking and pharmacokinetic models

Maass KF, Kulkarni C, Betts AM, Wittrup KD. Determination of Cellular Processing Rates for a Trastuzumab-Maytansinoid Antibody-Drug Conjugate (ADC) Highlights Key Parameters for ADC Design. AAPS J. 2016;18(3):635–646. [doi:10.1208/s12248-016-9892-3](https://doi.org/10.1208/s12248-016-9892-3).

Singh AP, Shah DK. Application of a PK-PD Modeling and Simulation-Based Strategy for Clinical Translation of Antibody-Drug Conjugates: a Case Study with Trastuzumab Emtansine (T-DM1). AAPS J. 2017;19(4):1054–1070. [doi:10.1208/s12248-017-0071-y](https://doi.org/10.1208/s12248-017-0071-y).

Menezes B, Cilliers C, Wessler T, Thurber GM, Linderman JJ. An Agent-Based Systems Pharmacology Model of the Antibody-Drug Conjugate Kadcyla to Predict Efficacy of Different Dosing Regimens. AAPS J. 2020;22(2):29. [doi:10.1208/s12248-019-0391-1](https://doi.org/10.1208/s12248-019-0391-1).

Betts A, Clark T, Jasper P, et al. Use of translational modeling and simulation for quantitative comparison of PF-06804103, a new generation HER2 ADC, with Trastuzumab-DM1. J Pharmacokinet Pharmacodyn. 2020;47(5):513–526. [doi:10.1007/s10928-020-09702-3](https://doi.org/10.1007/s10928-020-09702-3).

### Resources to inform possible model modifications

Ritchie M, Tchistiakova L, Scott N. Implications of receptor-mediated endocytosis and intracellular trafficking dynamics in the development of antibody drug conjugates. MAbs. 2013;5(1):13–21. [doi:10.4161/mabs.22854](https://doi.org/10.4161/mabs.22854).

This review discusses how trafficking of the targeted receptor (HER2 in our system) can influence ADC efficacy. Can you think of a way to change how the ADC binds to HER2 monomers and/or dimers that would change its trafficking?

Lyon RP, Setter JR, Bovee TD, et al. Self-hydrolyzing maleimides improve the stability and pharmacological properties of antibody-drug conjugates. Nat Biotechnol. 2014;32(10):1059–1062. [doi:10.1038/nbt.2968](https://doi.org/10.1038/nbt.2968).

This paper developed a new kind of chemical linker to attach cytotoxic drugs to antibodies, which slowed release of drug in circulation and thus reduced toxic side effects associated with premature payload release.

Loganzo F, Tan X, Sung M, et al. Tumor cells chronically treated with a trastuzumab-maytansinoid antibody-drug conjugate develop varied resistance mechanisms but respond to alternate treatments. Mol Cancer Ther. 2015;14(4):952–963. [doi:10.1158/1535-7163.MCT-14-0862](https://doi.org/10.1158/1535-7163.MCT-14-0862).

This paper showed that reduced HER2 expression was one mechanism of resistance in cancer cells chronically treated with a trastuzumab-maytansinoid ADC. Could you design your ADC so that it retains its ability to kill cancer cells even if they evolve to reduce their HER2 expression level?

Tsui CK, Barfield RM, Fischer CR, et al. CRISPR-Cas9 screens identify regulators of antibody-drug conjugate toxicity. Nat Chem Biol. 2019;15(10):949–958. [doi:10.1038/s41589-019-0342-2](https://doi.org/10.1038/s41589-019-0342-2).

This paper identified multiple biological regulators of toxicity of a HER2-targeted ADC expressed in cancer cells, offering opportunities for combination therapy strategies to improve ADC efficacy against their tumor targets.

Tsao LC, Wang JS, Ma X, et al. Effective extracellular payload release and immunomodulatory interactions govern the therapeutic effect of trastuzumab deruxtecan (T-DXd). Nat Commun. 2025;16(1):3167. [doi:10.1038/s41467-025-58266-8](https://doi.org/10.1038/s41467-025-58266-8).

In some settings, Enhertu can elicit an immune response to cancer, which contributes to its overall efficacy and the durability of patient responses. Can you think of a simple way to model the immune effects of the Enhertu ADC?

Kumagai K, Aida T, Tsuchiya Y, Kishino Y, Kai K, Mori K. Interstitial pneumonitis related to trastuzumab deruxtecan, a human epidermal growth factor receptor 2-targeting Ab-drug conjugate, in monkeys. Cancer Sci. 2020;111(12):4636–4645. [doi:10.1111/cas.14686](https://doi.org/10.1111/cas.14686).

This study implicates uptake of intact ADC by alveolar macrophages in lung toxicity in monkeys. What toxicity mechanisms could the base model miss when it compares designs using released-payload exposure in blood?